In [1]:
# ablation_run.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import copy
import time
import random
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# LOGGER SETUP (Checkpointing & Logging)
# ==========================================
class Logger(object):
    def __init__(self, filename="run.log"):
        self.terminal = sys.stdout
        os.makedirs(os.path.dirname(filename) or '.', exist_ok=True)
        self.log = open(filename, "a")
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
    def flush(self):
        self.terminal.flush()
        self.log.flush()

os.makedirs('./outputs', exist_ok=True)
sys.stdout = Logger('./outputs/run.log')

# ==========================================
# 1. LOCKED KFN ARCHITECTURE 
# ==========================================
class CosineLinear(nn.Module):
    def __init__(self, in_features, out_features, sigma=10.0):
        super(CosineLinear, self).__init__()
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.sigma = nn.Parameter(torch.Tensor([sigma]))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
    def forward(self, input):
        return self.sigma * F.linear(F.normalize(input, p=2, dim=1),
                                     F.normalize(self.weight, p=2, dim=1))

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                nn.BatchNorm2d(out_c)
            )
    def forward(self, x):
        return F.relu(self.bn1(self.conv1(x)) + self.shortcut(x))

class Specialist(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.final_conv = nn.Conv2d(256, 64, 1)
    def _make_layer(self, in_c, out_c, blocks, stride=1):
        layers = [ResidualBlock(in_c, out_c, stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_c, out_c))
        return nn.Sequential(*layers)
    def forward(self, x):
        return self.final_conv(self.pool(self.layer3(self.layer2(self.layer1(self.relu(self.bn1(self.conv1(x))))))))

class StructuralFusionModule(nn.Module):
    def __init__(self, in_channels, old_dim, new_dim, novelty_score):
        super().__init__()
        self.novelty_score = novelty_score
        self.has_expansion = new_dim > 0
        self.proj_reuse = nn.Sequential(
            nn.Conv2d(in_channels, old_dim, 1, bias=False),
            nn.BatchNorm2d(old_dim), nn.ReLU(), nn.Dropout(0.1)
        )
        self.gate_reuse = nn.Parameter(torch.tensor([0.0]))
        if self.has_expansion:
            self.proj_expand = nn.Sequential(
                nn.Conv2d(in_channels, new_dim, 1, bias=False),
                nn.BatchNorm2d(new_dim), nn.ReLU()
            )
    def forward(self, x, old_memory_detached):
        reuse_scale = max(0.1, 1.0 - self.novelty_score)
        delta_old = self.proj_reuse(x) * torch.sigmoid(self.gate_reuse) * reuse_scale
        feat_new = (self.proj_expand(x) - self.proj_expand(x).mean(dim=(2,3), keepdim=True)) if self.has_expansion else None
        return delta_old, feat_new

class GlobalModel(nn.Module):
    def __init__(self, n_specs, ch, n_classes, old_dim=64, new_dim=0, novelty_score=0.0):
        super().__init__()
        self.old_proj = nn.ModuleList([nn.Conv2d(ch, ch, 1) for _ in range(n_specs)])
        self.old_gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(ch, ch//4, 1), nn.ReLU(), nn.Conv2d(ch//4, ch, 1), nn.Sigmoid())
        self.old_bottleneck = nn.Sequential(nn.Conv2d(ch, old_dim, 1), nn.ReLU())
        self.old_dim = old_dim
        self.new_dim = new_dim
        self.fusion = StructuralFusionModule(ch, old_dim, new_dim, novelty_score) if new_dim > 0 else None
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = CosineLinear(old_dim + new_dim, n_classes)
    def forward_old(self, feats):
        p = self.old_proj[0](feats[0])
        return self.old_bottleneck(p * self.old_gate(p))
    def forward(self, feats):
        old_mem = self.forward_old(feats)
        if self.fusion:
            delta, f_new = self.fusion(feats[-1], old_mem.detach())
            final = torch.cat([old_mem + delta, f_new], dim=1) if f_new is not None else old_mem + delta
        else:
            final = old_mem
        W_old = self.classifier.weight[:5, :self.old_dim]
        W_new = self.classifier.weight[5:, :]
        logits_old = F.linear(F.normalize(self.pool(old_mem).flatten(1), p=2, dim=1),
                              F.normalize(W_old, p=2, dim=1)) * self.classifier.sigma
        logits_new = F.linear(F.normalize(self.pool(final).flatten(1), p=2, dim=1),
                              F.normalize(W_new, p=2, dim=1)) * self.classifier.sigma
        return torch.cat([logits_old, logits_new], dim=1), old_mem

class KFN(nn.Module):
    def __init__(self, n_classes=5, n_specs=1, old_dim=64, new_dim=0, novelty_score=0.0):
        super().__init__()
        self.specialists = nn.ModuleList([Specialist() for _ in range(n_specs)])
        self.global_model = GlobalModel(n_specs, 64, n_classes, old_dim, new_dim, novelty_score)
    def forward(self, x):
        return self.global_model([s(x) for s in self.specialists])

# ==========================================
# 2. EXPERIMENT UTILITIES & BULLETPROOF LOADER
# ==========================================
def set_deterministic(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def count_total_params(model):
    return sum(p.numel() for p in model.parameters())

def get_peak_memory(device):
    if torch.cuda.is_available() and getattr(device, "type", "") == 'cuda':
        max_mem = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
        torch.cuda.reset_peak_memory_stats(device)
        return max_mem
    return 0.0

def evaluate(model, loader, device):
    model.eval(); correct = 0; total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            # use mixed precision safe eval with cuda AMP
            if device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    logits, _ = model(x)
            else:
                logits, _ = model(x)
            correct += (logits.argmax(1) == y).sum().item(); total += y.size(0)
    return 100 * correct / total

def get_kaggle_dataset_robust(transform_train, transform_test):
    data_root = './data'
    found_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'data_batch_1' in files:
            found_path = root
            break
    if found_path:
        if os.path.basename(found_path) == 'cifar-10-batches-py':
            data_root = os.path.dirname(found_path)
        else:
            os.makedirs('./data/cifar-10-batches-py', exist_ok=True)
            for f in os.listdir(found_path):
                src = os.path.join(found_path, f)
                dst = os.path.join('./data/cifar-10-batches-py', f)
                if not os.path.exists(dst):
                    try: os.symlink(src, dst)
                    except Exception: pass
            data_root = './data'
    try:
        train_ds = torchvision.datasets.CIFAR10(root=data_root, train=True, download=False, transform=transform_train)
        test_ds = torchvision.datasets.CIFAR10(root=data_root, train=False, download=False, transform=transform_test)
        print(f"✅ Offline Data Loaded successfully from: {data_root}")
        return train_ds, test_ds
    except Exception as e:
        print(f"\n❌ CRITICAL ERROR: Could not load dataset. Details: {e}")
        sys.exit(1)

# ==========================================
# 3. EXPERIMENT CONTROLLER (Ablation Logic)
# ==========================================
def run_kfn_experiment(mode, seed, loaders, epochs, device):
    set_deterministic(seed)
    l_A, l_B, t_A, t_B, replay_loader = loaders
    ep_p1, ep_p2, ep_p3, ep_p4 = epochs

    get_peak_memory(device)

    # Mixed precision scaler
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

    # --- PHASE 1: Base Model (Task A) ---
    model = KFN(n_classes=5, n_specs=1).to(device)
    opt1 = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for _ in range(ep_p1):
        model.train()
        for x, y in l_A:
            x, y = x.to(device), y.to(device)
            opt1.zero_grad()
            if device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    out, _ = model(x)
                    loss = F.cross_entropy(out, y)
                scaler.scale(loss).backward()
                scaler.step(opt1)
                scaler.update()
            else:
                out, _ = model(x)
                loss = F.cross_entropy(out, y)
                loss.backward()
                opt1.step()

    acc_A_init = evaluate(model, t_A, device)
    teacher = copy.deepcopy(model).eval()
    base_params = count_total_params(model)

    # --- PHASE 2: Specialist (Task B) ---
    start_p2 = time.time()
    spec = Specialist().to(device)
    head = nn.Sequential(
        nn.Linear(64, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 5)
    ).to(device)
    opt2 = torch.optim.AdamW(list(spec.parameters()) + list(head.parameters()), lr=1e-3, weight_decay=1e-4)
    for _ in range(ep_p2):
        spec.train()
        for x, y in l_B:
            x, y = x.to(device), (y-5).to(device)
            opt2.zero_grad()
            if device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    feats = F.adaptive_avg_pool2d(spec(x), 1).flatten(1)
                    loss = F.cross_entropy(head(feats), y)
                scaler.scale(loss).backward()
                scaler.step(opt2)
                scaler.update()
            else:
                feats = F.adaptive_avg_pool2d(spec(x), 1).flatten(1)
                loss = F.cross_entropy(head(feats), y)
                loss.backward()
                opt2.step()
    time_p2 = time.time() - start_p2
    mem_p2 = get_peak_memory(device)

    # --- EXPANSION ALGORITHM ---
    new_dim = 96 if mode != "REPLAY_ONLY" else 0
    new_model = KFN(n_classes=10, n_specs=2, old_dim=64, new_dim=new_dim, novelty_score=0.99).to(device)
    new_model.specialists[0].load_state_dict(model.specialists[0].state_dict())
    new_model.specialists[1].load_state_dict(spec.state_dict())
    new_model.global_model.old_proj[0].load_state_dict(model.global_model.old_proj[0].state_dict())
    new_model.global_model.old_gate.load_state_dict(model.global_model.old_gate.state_dict())
    new_model.global_model.old_bottleneck.load_state_dict(model.global_model.old_bottleneck.state_dict())
    with torch.no_grad():
        new_model.global_model.classifier.weight[:5, :64] = model.global_model.classifier.weight
        new_model.global_model.classifier.sigma.data = model.global_model.classifier.sigma.data.clone()
        new_model.global_model.classifier.weight[:5, 64:].zero_()
    model = new_model

    expanded_params = count_total_params(model)
    added_params = expanded_params - base_params

    # --- PHASE 3: Integration ---
    for n, p in model.named_parameters(): p.requires_grad = False
    model.global_model.classifier.weight.requires_grad = True
    if model.global_model.fusion:
        model.global_model.fusion.requires_grad_(True)

    start_p3 = time.time()
    rep_iter = iter(replay_loader)
    opt3 = None

    for ep in range(ep_p3):
        rebuild_opt = False
        if mode != "NO_GRADUAL":
            if ep == 0:
                model.global_model.old_gate.requires_grad_(False)
                model.global_model.old_bottleneck.requires_grad_(True)
                rebuild_opt = True
            elif ep == ep_p3 // 2:
                model.global_model.old_gate.requires_grad_(True)
                model.global_model.old_bottleneck.requires_grad_(True)
                rebuild_opt = True
        else:
            if ep == 0:
                rebuild_opt = True

        if rebuild_opt:
            active_params = []
            for n, p in model.named_parameters():
                if p.requires_grad:
                    # expansion & new specialist get larger LR
                    if 'fusion' in n or 'classifier' in n or 'specialists.1' in n:
                        active_params.append({'params': p, 'lr': 2e-3})
                    else:
                        active_params.append({'params': p, 'lr': 5e-4})
            opt3 = torch.optim.AdamW(active_params, weight_decay=1e-4)

        model.train()
        for x_B, y_B in l_B:
            x_B, y_B = x_B.to(device), y_B.to(device)
            try:
                x_A, y_A = next(rep_iter)
            except StopIteration:
                rep_iter = iter(replay_loader)
                x_A, y_A = next(rep_iter)
            
            x_A = x_A.to(device)
            y_A = y_A.to(device)

            opt3.zero_grad()
            if device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    logits_B, _ = model(x_B)
                    l_b_task = F.cross_entropy(logits_B, y_B)
                    l_replay = 0.0; l_dist = 0.0; l_feat = 0.0
                    if mode != "FUSION_ONLY":
                        logits_A, m_A = model(x_A)
                        l_replay = F.cross_entropy(logits_A, y_A)
                        with torch.no_grad():
                            logits_T, m_T = teacher(x_A)
                        l_dist = F.kl_div(F.log_softmax(logits_A[:, :5]/2.0, 1),
                                          F.softmax(logits_T[:, :5]/2.0, 1), reduction='batchmean') * 4.0
                        l_feat = F.mse_loss(m_A, m_T)
                    loss = 2.0*l_b_task + 0.6*l_replay + 1.0*l_dist + 1.2*l_feat
                scaler.scale(loss).backward()
                scaler.step(opt3)
                scaler.update()
            else:
                logits_B, _ = model(x_B)
                l_b_task = F.cross_entropy(logits_B, y_B)
                l_replay = 0.0; l_dist = 0.0; l_feat = 0.0
                if mode != "FUSION_ONLY":
                    logits_A, m_A = model(x_A)
                    l_replay = F.cross_entropy(logits_A, y_A)
                    with torch.no_grad():
                        logits_T, m_T = teacher(x_A)
                    l_dist = F.kl_div(F.log_softmax(logits_A[:, :5]/2.0, 1),
                                      F.softmax(logits_T[:, :5]/2.0, 1), reduction='batchmean') * 4.0
                    l_feat = F.mse_loss(m_A, m_T)
                loss = 2.0*l_b_task + 0.6*l_replay + 1.0*l_dist + 1.2*l_feat
                loss.backward()
                opt3.step()

        # periodic checkpoint
        if (ep + 1) % 5 == 0:
            try:
                torch.save({
                    'model': model.state_dict(),
                    'opt3': opt3.state_dict() if opt3 is not None else None
                }, f'./outputs/kfn_{mode}_seed{seed}_ep{ep+1}.pth')
            except Exception as e:
                print(f"Warning: checkpoint save failed: {e}")

    time_p3 = time.time() - start_p3
    mem_p3 = get_peak_memory(device)

    # --- PHASE 4: Bias Correction ---
    start_p4 = time.time()
    for p in model.parameters(): p.requires_grad = False
    model.global_model.classifier.weight.requires_grad = True
    opt4 = torch.optim.AdamW(model.global_model.classifier.parameters(), lr=5e-5)
    for _ in range(ep_p4):
        model.train()
        for x, y in replay_loader:
            x, y = x.to(device), y.to(device)
            opt4.zero_grad()
            if device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    out, _ = model(x)
                    loss = F.cross_entropy(out, y)
                scaler.scale(loss).backward()
                scaler.step(opt4)
                scaler.update()
            else:
                out, _ = model(x)
                loss = F.cross_entropy(out, y)
                loss.backward()
                opt4.step()
    time_p4 = time.time() - start_p4
    mem_p4 = get_peak_memory(device)

    acc_A_final = evaluate(model, t_A, device)
    acc_B_final = evaluate(model, t_B, device)

    return {
        "acc_A_init": acc_A_init, "acc_A_final": acc_A_final, "acc_B_final": acc_B_final,
        "time_p2": time_p2, "time_p3": time_p3, "time_p4": time_p4,
        "inc_time": time_p2 + time_p3 + time_p4,
        "mem_p2": mem_p2, "mem_p3": mem_p3, "mem_p4": mem_p4,
        "base_p": base_params, "exp_p": expanded_params, "added_p": added_params
    }

def run_full_retrain_baseline(train_ds, epochs, device, bs):
    l_full = DataLoader(train_ds, batch_size=bs, shuffle=True, pin_memory=False)
    model = KFN(n_classes=10, n_specs=2, new_dim=96).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
    get_peak_memory(device)
    start = time.time()
    for _ in range(epochs):
        model.train()
        for x, y in l_full:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            if device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    out, _ = model(x)
                    loss = F.cross_entropy(out, y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                out, _ = model(x)
                loss = F.cross_entropy(out, y)
                loss.backward()
                opt.step()
    return time.time() - start, get_peak_memory(device)

# ==========================================
# 4. EXPERIMENT EXECUTION & REPORTING
# ==========================================
def run_all():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Environment Configured: {device}")

    stats = ((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))

    # stronger augmentation for training
    t_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(*stats)
    ])
    t_test = transforms.Compose([transforms.ToTensor(), transforms.Normalize(*stats)])
    train_ds, test_ds = get_kaggle_dataset_robust(t_train, t_test)

    # hyperparameters (tuned for ablations run)
    EPOCHS = (10, 20, 40, 5)  # P1, P2, P3, P4
    REPLAY_SIZE = 5000
    BS = 256

    loaders = (
        DataLoader(Subset(train_ds, [i for i, l in enumerate(train_ds.targets) if l < 5]), BS, shuffle=True, pin_memory=False),
        DataLoader(Subset(train_ds, [i for i, l in enumerate(train_ds.targets) if l >= 5]), BS, shuffle=True, pin_memory=False),
        DataLoader(Subset(test_ds, [i for i, l in enumerate(test_ds.targets) if l < 5]), BS, pin_memory=False),
        DataLoader(Subset(test_ds, [i for i, l in enumerate(test_ds.targets) if l >= 5]), BS, pin_memory=False),
        DataLoader(Subset(train_ds, [i for i, l in enumerate(train_ds.targets) if l < 5][:REPLAY_SIZE]), BS, shuffle=True, pin_memory=False)
    )

    print(f"\n[1] Running Compute Baseline (Full Retrain for {EPOCHS[2]} epochs)...")
    baseline_time, baseline_mem = run_full_retrain_baseline(train_ds, EPOCHS[2], device, BS)

    # run all seeds (unconditional)
    print("\n[2] Running Multi-Seed Experiments (FULL Mode)...")
    seeds = [0, 1, 2, 3, 4]
    full_results = []
    for s in seeds:
        print(f" -> Seed {s}")
        full_results.append(run_kfn_experiment("FULL", s, loaders, EPOCHS, device))

    # ablations (unconditional)
    print("\n[3] Running Ablations...")
    ablations = {}
    for mode in ["REPLAY_ONLY", "FUSION_ONLY", "NO_GRADUAL"]:
        print(f" -> Mode: {mode}")
        ablations[mode] = run_kfn_experiment(mode, 42, loaders, EPOCHS, device)

    # reporting
    def calc_metrics(r):
        return {
            "ret": r['acc_A_final'] / r['acc_A_init'],
            "forg": r['acc_A_init'] - r['acc_A_final'],
            "bwt": r['acc_A_final'] - r['acc_A_init']
        }

    print("\n" + "="*80)
    print("SECTION A: Main Results (Mean ± Std over 5 Seeds)")
    print("="*80)
    for k, label in [("acc_A_init", "Task A Init"), ("acc_A_final", "Task A Final"), ("acc_B_final", "Task B Final")]:
        vals = [r[k] for r in full_results]
        print(f"{label:15}: {np.mean(vals):.2f}% ± {np.std(vals):.2f}")

    ret_vals = [(r['acc_A_final']/r['acc_A_init'])*100 for r in full_results]
    forg_vals = [r['acc_A_init'] - r['acc_A_final'] for r in full_results]
    bwt_vals = [r['acc_A_final'] - r['acc_A_init'] for r in full_results]

    print(f"Retention:      {np.mean(ret_vals):.2f}% ± {np.std(ret_vals):.2f}")
    print(f"Forgetting:     {np.mean(forg_vals):.2f} ± {np.std(forg_vals):.2f}")
    print(f"Backward Trans: {np.mean(bwt_vals):.2f} ± {np.std(bwt_vals):.2f}")

    print("\n" + "="*80)
    print("SECTION B: Ablation Table")
    print("="*80)
    print(f"{'Mode':15} | {'Task A':10} | {'Task B':10} | {'Retention':10} | {'Forgetting':10}")
    print("-" * 80)

    f_mean_A = np.mean([r['acc_A_final'] for r in full_results])
    f_mean_B = np.mean([r['acc_B_final'] for r in full_results])
    print(f"{'FULL':15} | {f_mean_A:<10.2f} | {f_mean_B:<10.2f} | {np.mean(ret_vals):<9.1f}% | {np.mean(forg_vals):<10.2f}")

    for mode, r in ablations.items():
        m = calc_metrics(r)
        print(f"{mode:15} | {r['acc_A_final']:<10.2f} | {r['acc_B_final']:<10.2f} | {m['ret']*100:<9.1f}% | {m['forg']:<10.2f}")

    print("\n" + "="*80)
    print("SECTION C: Parameter Growth")
    print("="*80)
    rep = full_results[0]
    print(f"Base Parameters:        {rep['base_p']:,}")
    print(f"Expanded Parameters:    {rep['exp_p']:,}")
    print(f"Added Parameters:       {rep['added_p']:,}")
    print(f"% Increase:             {(rep['added_p']/rep['base_p'])*100:.2f}%")

    print("\n" + "="*80)
    print("SECTION D: Compute & Memory Comparison")
    print("="*80)
    kfn_time = np.mean([r["inc_time"] for r in full_results])
    peak_kfn_mem = np.mean([max(r["mem_p2"], r["mem_p3"], r["mem_p4"]) for r in full_results])

    print(f"KFN Phase 2 Time (Specialist): {np.mean([r['time_p2'] for r in full_results]):.2f}s")
    print(f"KFN Phase 3 Time (Integrate):  {np.mean([r['time_p3'] for r in full_results]):.2f}s")
    print(f"KFN Phase 4 Time (Bias Corr):  {np.mean([r['time_p4'] for r in full_results]):.2f}s")
    print("-" * 80)
    print(f"Total Incremental Time:        {kfn_time:.2f}s")
    print(f"Full Retrain Time ({EPOCHS[2]} epochs): {baseline_time:.2f}s")
    print(f"Speedup Factor:                {baseline_time/kfn_time:.2f}x")
    print("-" * 80)
    print(f"KFN Peak Memory:               {peak_kfn_mem:.2f} MB")
    print(f"Baseline Peak Memory:          {baseline_mem:.2f} MB")

    print("\n" + "="*80)
    print("SECTION E: Efficiency Metrics")
    print("="*80)
    total_m_params = rep['exp_p'] / 1e6
    avg_acc = (f_mean_A + f_mean_B) / 2.0
    print(f"Accuracy per Million Params:   {avg_acc / total_m_params:.2f}")
    print(f"Accuracy per Second:           {avg_acc / kfn_time:.2f}")
    print("="*80)

if __name__ == "__main__":
    run_all()

🚀 Environment Configured: cuda
✅ Offline Data Loaded successfully from: /kaggle/input/cifar-10

[1] Running Compute Baseline (Full Retrain for 40 epochs)...

[2] Running Multi-Seed Experiments (FULL Mode)...
 -> Seed 0
 -> Seed 1
 -> Seed 2
 -> Seed 3
 -> Seed 4

[3] Running Ablations...
 -> Mode: REPLAY_ONLY
 -> Mode: FUSION_ONLY
 -> Mode: NO_GRADUAL

SECTION A: Main Results (Mean ± Std over 5 Seeds)
Task A Init    : 78.98% ± 5.33
Task A Final   : 68.64% ± 3.16
Task B Final   : 77.56% ± 2.50
Retention:      87.08% ± 3.26
Forgetting:     10.34 ± 3.11
Backward Trans: -10.34 ± 3.11

SECTION B: Ablation Table
Mode            | Task A     | Task B     | Retention  | Forgetting
--------------------------------------------------------------------------------
FULL            | 68.64      | 77.56      | 87.1     % | 10.34     
REPLAY_ONLY     | 57.78      | 70.16      | 71.3     % | 23.26     
FUSION_ONLY     | 0.00       | 93.34      | 0.0      % | 81.04     
NO_GRADUAL      | 62.22      | 82